# PDF Embedding Pipeline

Extract PDF pages, create semantic chunks, generate normalized 384-dimensional embeddings, upload them to Qdrant in batches of 100, and validate the collection.

In [ ]:
from pathlib import Path

from embedding_generator import create_embedding_model, generate_embeddings
from pdf_extractor import extract_pdf_pages
from quadrant_client import (
    QDRANT_BATCH_SIZE,
    QDRANT_COLLECTION_NAME,
    ensure_collection,
    get_qdrant_client,
    upsert_embedded_chunks,
    verify_collection,
)
from semantic_chunker import semantic_chunk_pages


In [ ]:
pdf_paths = sorted(Path('data/docs').glob('*.pdf'))
pages = [page for path in pdf_paths for page in extract_pdf_pages(path)]
chunks = semantic_chunk_pages(pages)
print(f'{len(pdf_paths)} documents, {len(chunks)} semantic chunks')


In [ ]:
embedding_model = create_embedding_model()
embedded_chunks = generate_embeddings(chunks, embedding_model)
assert len(embedded_chunks) == len(chunks)
assert all(len(chunk['embedding']) == 384 for chunk in embedded_chunks)
print(f'{len(embedded_chunks)} normalized embeddings generated')


In [ ]:
client = get_qdrant_client()
ensure_collection(client, QDRANT_COLLECTION_NAME)
uploaded = upsert_embedded_chunks(
    client,
    embedded_chunks,
    collection_name=QDRANT_COLLECTION_NAME,
    batch_size=QDRANT_BATCH_SIZE,
)
print(f'{uploaded} points uploaded in batches of {QDRANT_BATCH_SIZE}')


In [ ]:
report = verify_collection(
    client,
    collection_name=QDRANT_COLLECTION_NAME,
    expected_point_count=len(chunks),
)
assert report['valid']
assert report['point_count'] == len(chunks)
report
